# Stage 3 — ESCO Occupation Classification (Batch API)
## Notebook 3.3 of 7 — Extract Classification Results

**Role in the pipeline:**
Reads the Batch API output JSONL files downloaded by notebook 3.2, parses the function-call arguments to extract the LLM-assigned ESCO occupation code and title, and merges them into the Stage 2 output DataFrames. The resulting pickles (with `esco_code` and `esco_title` columns added) are saved to `STAGE3_RESULT_PATH`.

Also computes per-file statistics on how many vacancies received a valid classification and logs them to the tracker. This information is used by notebook 3.4 to separate complete from missing records.

**Position in Stage 3 sequence:**
1. 3.1 — Create batch input files
2. 3.2 — Submit batch jobs and monitor status
3. ▶ **3.3 — Extract classification results** ← *you are here*
4. 3.4 — Split missing and complete classifications
5. 3.5 — Create batch input files for missing records
6. 3.6 — Submit batch jobs for missing records *(run after Batch API completes)*
7. 3.7 — Extract results for missing records

**Prerequisites:**
- Notebook 3.2 completed — `output_batch_status == 'complete'` for all rows

**Documentation:** [Notebook guide](README.md) · [Stage data description](../../data/data-pipeline/stage_03/README_data.md)


## 3.3.1. Load packages and configuration

Loads standard libraries and the shared `general` config module.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g
g.clean_memory()

Imports Stage 2/3 processing modules and pandas.

In [ ]:
import stage2 as st2
import stage3 as st3
import pandas as pd

## 3.3.2. Load stage 3 logs

Reads the Stage 3 tracker. The commented-out lines show how to reset `result_status` and `result_path` columns for a full re-extraction from scratch (use only if the result pickles need to be regenerated).

Loads the tracker, with optional commented-out reset lines for `result_status` and `result_path`.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
process_df

## 3.3.3. Read output batches and add data to results

Extraction loop: for each row where `output_batch_status == 'complete'` but `result_status != 'complete'`:

1. Loads the Stage 2 output pickle (all vacancy columns).
2. Calls `extract_esco_codes()` to parse the Batch API output JSONL and return a DataFrame with `id`, `esco_code`, `esco_title`.
3. Merges the ESCO columns onto the Stage 2 data via a left-join on `id`. Vacancies whose request failed will have `NaN` in the ESCO columns (these become the "missing" records handled by notebooks 3.4–3.7).
4. Saves the merged pickle to `STAGE3_RESULT_PATH` and marks `result_status = 'complete'`.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)

for _,row in process_df.iterrows():
    if row["output_batch_status"] == "complete" and row["result_status"] != "complete":
        result_df = pd.read_pickle(row["extract_path"])
        esco_df = st3.extract_esco_codes(row["output_batch_path"])
        result_df['id'] = result_df['id'].astype(str)

        result_df = pd.merge(result_df, esco_df, on='id', how='left')

        result_path = os.path.join(g.Config.STAGE3_RESULT_PATH, row["input_file"]  + ".pkl")
        result_df.to_pickle(result_path)
        process_df.loc[_, "result_path"] = result_path
        process_df.loc[_, "result_status"] = "complete"

        process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)
        print("Added result file for: ", row["input_file"])

Displays the last result DataFrame extracted by the loop above — useful for a quick sanity check on the merged columns.

In [ ]:
result_df

## 3.3.4. Check completion statistics

Prints a summary of the processing pipeline progress across all tracked files:
- Batch input files created
- API jobs completed
- Output batch files downloaded
- Result pickles extracted

All four counts should reach 100% before proceeding to notebook 3.4.

In [ ]:
input_created= (process_df["input_batch_status"] == "created").sum()
t = process_df.shape[0]
jobs_completed = (process_df["job_status"] == "completed").sum()
output_complete = (process_df["output_batch_status"] == "complete").sum()
result_complete = (process_df["result_status"] == "complete").sum()

print(f"Total created input batches {input_created} of {t}:\t\t{round(100*input_created/t, 2)} %")
print(f"Total completed jobs {jobs_completed} of {t}:\t\t\t{round(100*jobs_completed/t, 2)} %")
print(f"Total created output batches {output_complete} of {t}:\t{round(100*output_complete/t, 2)} %")
print(f"Total complete results {result_complete} of {t}:\t\t\t{round(100*result_complete/t, 2)} %")

Per-file missing count check: iterates over all result pickles, counts how many vacancies have a missing or very short `esco_title` (length < 3), and stores the `missing` count, `total` count, and `missing_percent` in a summary DataFrame. These become the records targeted by notebooks 3.5–3.7.

In [ ]:
stats_df = pd.DataFrame(
    {"input_file": process_df["input_file"], "result_path": process_df["result_path"], "missing": None, "total": None,
     "missing_percent": None})

for i, row in stats_df.iterrows():
    print("Working on: ", row["input_file"], "")
    try:
        df = pd.read_pickle(row["result_path"])
    except Exception:
        print(f"Error reading file {row['result_path']}")
        continue

    total = len(df)
    if total == 0:
        stats_df.loc[i, ["total", "missing", "missing_percent"]] = [0, 0, None]
        continue

    if "esco_title" in df.columns:
        ser = df["esco_title"]
        missing_mask = ser.isna() | (ser.astype(str).str.len() < 3)
        missing = int(missing_mask.sum())
    else:
        missing = total  # if column absent, consider all missing

    stats_df.loc[i, "total"] = total
    stats_df.loc[i, "missing"] = missing
    stats_df.loc[i, "missing_percent"] = (missing / total) if total else None

stats_df

Shows the column names present in the tracker at this point — useful for verifying which columns are available for the next stage.

In [ ]:
process_df.columns

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.